#Исследуем соответствуют ли типы данных в датасете необходимым.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df=pd.read_excel('/content/cian_full_data.xlsx')
df.head()

,url,address,price,auction,rooms_count,new_building,apartment,total_area,living_area,kitchen_area,...,lat,lng,minor_owners,maternity_capital,publication_date,request_time,photos_url,seller,finishing,ramp
0,https://www.cian.ru/sale/flat/324994191/,"Москва, ЮАО, р-н Даниловский, м. Автозаводская...",5350000,Н,Студия,Вторичка,Апартаменты,16.0,10.0,5.0,...,55.703885,37.652768,Н,Н,2025-12-19 12:29:00,2025-12-20 03:40:29,https://images.cdn-cian.ru/images/kvartira-mos...,NaN,NaN,NaN
1,https://www.cian.ru/sale/flat/320317503/,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",5990000,Н,Студия,Вторичка,Апартаменты,16.0,13.5,NaN,...,55.859322,37.515874,NaN,NaN,2025-12-19 16:25:00,2025-12-20 03:40:32,https://images.cdn-cian.ru/images/kvartira-mos...,NaN,NaN,NaN
2,https://www.cian.ru/sale/flat/322151830/,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",5990000,Н,Студия,Вторичка,Апартаменты,15.2,12.7,NaN,...,55.859322,37.515874,NaN,NaN,2025-12-19 19:55:00,2025-12-20 03:40:38,https://images.cdn-cian.ru/images/kvartira-mos...,NaN,NaN,NaN
3,https://www.cian.ru/sale/flat/323600775/,"Москва, ТАО (Троицкий), пос. Рогово, улица Шко...",5500000,Н,1,Вторичка,NaN,32.6,NaN,7.5,...,55.212647,37.072573,NaN,NaN,2025-12-19 18:22:00,2025-12-20 03:40:41,https://images.cdn-cian.ru/images/2690280423-1...,NaN,NaN,NaN
4,https://www.cian.ru/sale/flat/324712040/,"Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, П...",5600000,Н,Студия,Вторичка,NaN,20.0,12.0,3.0,...,55.704996,37.922335,NaN,NaN,2025-12-18 18:26:00,2025-12-20 03:40:44,https://images.cdn-cian.ru/images/2725644747-1...,NaN,NaN,NaN


In [3]:
def overview(df):
    exclude_cols = ['url', 'description', 'photos_url', 'index', 'price']
    text_df = df.select_dtypes(include = ['object', 'string'])
    cols_to_analyze = [c for c in text_df.columns if c not in exclude_cols]

    fin_df = []
    total_rows = len(df)

    for col in cols_to_analyze:
        col_series = df[col]
        unique_values = col_series.dropna().unique()

        n_nan = col_series.isna().sum()
        n_filled = total_rows - n_nan
        n_unique = len(unique_values)
        zap = (n_unique/n_filled*100) if n_filled > 0 else 0

        examples = list(unique_values[:5])
        examples_str = " || ".join(map(str, examples))

        fin_df.append({"Признак": col,
            "Заполнено": n_filled,
            "Пропущено": n_nan,
            "Уникальных": n_unique,
            "Разнообразие в проц": round(zap, 2),
            "Примеры": examples_str})

    return pd.DataFrame(fin_df).set_index("Признак")

In [4]:
overview(df)

,Заполнено,Пропущено,Уникальных,Разнообразие в проц,Примеры
Признак,,,,,
address,63365,0,16511,26.06,"Москва, ЮАО, р-н Даниловский, м. Автозаводская..."
auction,63365,0,1,0.00,Н
rooms_count,63365,0,9,0.01,Студия || 1 || 2 || 3 || Доля
new_building,63365,0,2,0.00,Вторичка || Новостройка
apartment,7367,55998,2,0.03,Апартаменты || Пентхаус
view_from_the_windows,35772,27593,3,0.01,Во двор || На улицу || На улицу и двор
repair,35322,28043,4,0.01,Евроремонт || Косметический || Без ремонта || ...
construction_series,23607,39758,96,0.41,Индивидуальный проект || П-3М || 1-511 || ГМС-...
floor_type,23102,40263,3,0.01,Железобетонные || Деревянные || Смешанные


Как видим, некоторые признаки имеют излишнюю уникальность, которая может негативно повлиять на точность модели.
В нашем случае очевидно, что признаки:

1.   metro_info
2.   highway_info
3.   seller
4.   address

могут быть разбиты на несколько признаков, объекты которых будут значительно более подходящими для категориальных признаков по характеру значений - количество уникальных значений будет значительно меньшим, а также некоторые признаки выделенные будут числовыми - к примеру, расстояние до ближайшего шоссе или метро.

Начнем с анализа содержимого адреса.

In [5]:
df.address.unique()[:30]

array(['Москва, ЮАО, р-н Даниловский, м. Автозаводская, Автозаводская улица, 19К1',
       'Москва, САО, р-н Головинский, м. Моссельмаш, Сенежская улица, 3',
       'Москва, ТАО (Троицкий), пос. Рогово, улица Школьная, 17',
       'Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, Покровская улица, 31',
       'Москва, ТАО (Троицкий), пос. ЛМС, Центральный мкр, 7',
       'Москва, ВАО, р-н Измайлово, м. Измайловская, 4-я Парковая улица, 24',
       'Москва, ТАО (Троицкий), м. Троицк, с. Былово, Былово кп, улица Радужная, 2',
       'Москва, САО, р-н Ховрино, м. Грачёвская, Клинская улица, 8С1',
       'Москва, ТАО (Троицкий), Витро Вилладж База отдыха, к4',
       'Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, Рождественская улица, 21к5',
       'Москва, ТАО (Троицкий), Витро Вилладж База отдыха, к7',
       'Москва, ТАО (Троицкий), м. Троицк, Троицк, улица Городская, 20',
       'Москва, ВАО, р-н Преображенское, м. Электрозаводская, 1-й Электрозаводский переулок, 4',
       'Москва, ТАО (Тр

Очевидно логично выделить из адреса три признака: метро указанное в адресе, округ и район. Число уникальных объектов в данных признаках будет достаточно малым.

In [6]:
def parse_address(addr):
    if not isinstance(addr, str):
        return pd.Series([np.nan, np.nan, np.nan])

    s = addr.lower()

    okrug = np.nan
    district = np.nan
    metro = np.nan

    parts = [p.strip() for p in s.split(",")]

    if len(parts) > 1:
        okrug = parts[1]

    for p in parts:
        if p.startswith("р-н"):
            district = p
        elif p.startswith("м."):
            metro = p.replace("м.", "").strip()

    return pd.Series([okrug, district, metro])

In [7]:
df[["okrug", "district", "metro_adress"]] = df["address"].apply(parse_address)

In [8]:
df[["address","okrug", "district", "metro_adress"]].head(10)

,address,okrug,district,metro_adress
0,"Москва, ЮАО, р-н Даниловский, м. Автозаводская...",юао,р-н даниловский,автозаводская
1,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",сао,р-н головинский,моссельмаш
2,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",сао,р-н головинский,моссельмаш
3,"Москва, ТАО (Троицкий), пос. Рогово, улица Шко...",тао (троицкий),NaN,NaN
4,"Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, П...",ювао,р-н некрасовка,некрасовка
5,"Москва, ТАО (Троицкий), пос. ЛМС, Центральный ...",тао (троицкий),NaN,NaN
6,"Москва, ВАО, р-н Измайлово, м. Измайловская, 4...",вао,р-н измайлово,измайловская
7,"Москва, ТАО (Троицкий), м. Троицк, с. Былово, ...",тао (троицкий),NaN,троицк
8,"Москва, САО, р-н Ховрино, м. Грачёвская, Клинс...",сао,р-н ховрино,грачёвская
9,"Москва, ТАО (Троицкий), Витро Вилладж База отд...",тао (троицкий),NaN,NaN


Продолжим. Разделим признак продавца на новые признаки: тип продавца и уникальное имя.

In [9]:
df.seller.unique()

array([nan, 'Агентство недвижимости, Ордер',
       'Автор объявления, ID 135186715', ...,
       'Автор объявления, Ирина Иванова',
       'Агентство недвижимости, New Day Realty',
       'Автор объявления, Владислав Дворов'], dtype=object)

In [10]:
def parse_seller(cell):
    if not isinstance(cell, str) or cell.strip() == "":
        return pd.Series([None, None])
    parts = cell.split(",", 1)
    seller_type = parts[0].strip()
    seller_name = parts[1].strip() if len(parts)>1 else None
    return pd.Series([seller_type, seller_name])

df[["seller_type", "seller_name"]] = df["seller"].apply(parse_seller)

In [11]:
df[["seller","seller_type", "seller_name"]].dropna().head(5)

,seller,seller_type,seller_name
37499,"Агентство недвижимости, Ордер",Агентство недвижимости,Ордер
37500,"Автор объявления, ID 135186715",Автор объявления,ID 135186715
37501,"Автор объявления, ID 135186715",Автор объявления,ID 135186715
37502,"Автор объявления, ID 135186715",Автор объявления,ID 135186715
37503,"Автор объявления, ID 135186715",Автор объявления,ID 135186715


Просмотрим подробнее информацию о метро

In [12]:
df["metro_info"].head(10)

,metro_info
0,"Автозаводская 5 мин. (пешком), ЗИЛ 9 мин. (пеш..."
1,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин..."
2,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин..."
3,NaN
4,"Некрасовка 3 мин. (пешком), Лухмановская 18 ми..."
5,NaN
6,"Измайловская 12 мин. (пешком), Первомайская 12..."
7,"Троицк откроется в 2026 15 мин. (на машине), В..."
8,"Беломорская 18 мин. (пешком), Грачёвская 11 ми..."
9,NaN


Разобьем информацию о метро по аналогии на признаки: количество станций поблизости, ближайшая по скорости дороги, как добраться до ближайшей - на машине или пешком, и открыта ли ближайшая.

In [13]:
def parse_transport(cell):
    if not isinstance(cell, str) or cell.strip() == "":
        return pd.Series([None, None, None, 0, "нет"])

    pattern = r"(.+?)\s+(\d+)\s+мин\.\s+\((пешком|на машине)\)"
    matches = re.findall(pattern, cell)

    if not matches:
        return pd.Series([None, None, None, 0, "нет"])

    st, mins, mode = matches[0]

    st_name = st.strip().strip(", ")

    if "откроется в" in st_name:
        st_name = re.split(r"\s*откроется в\s*.*", st_name)[0].strip(", ")
        nearest_station_open = "нет"
    else:
        nearest_station_open = "да"

    return pd.Series([mode,
        int(mins),
        st_name,
        len(matches),
        nearest_station_open])

df[["fastest_mode",
    "fastest_minutes",
    "nearest_station",
    "stations_nearby",
    "nearest_station_open"]] = df["metro_info"].apply(parse_transport)


In [14]:
df[[ "metro_info",
    "fastest_mode",
    "fastest_minutes",
    "nearest_station",
    "stations_nearby",
    "nearest_station_open"]].dropna().head(5)

,metro_info,fastest_mode,fastest_minutes,nearest_station,stations_nearby,nearest_station_open
0,"Автозаводская 5 мин. (пешком), ЗИЛ 9 мин. (пеш...",пешком,5.0,Автозаводская,3,да
1,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин...",пешком,10.0,Моссельмаш,3,да
2,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин...",пешком,10.0,Моссельмаш,3,да
4,"Некрасовка 3 мин. (пешком), Лухмановская 18 ми...",пешком,3.0,Некрасовка,3,да
6,"Измайловская 12 мин. (пешком), Первомайская 12...",пешком,12.0,Измайловская,3,да


Закончим выделение признаков в более дискретные или числовые на разделении информации о шоссе поблизости: выделим в числовую переменную расстояние до мкада, название ближайшего шоссе и количество шоссе поблизости.

In [15]:
def parse_roads(cell):
    if not isinstance(cell, str) or cell.strip() == "":
        return pd.Series([None, None, 0])

    pattern = r"([^,]+?)(?:,\s*(\d+)\s*км от МКАД)?(?=,|$)"
    matches = re.findall(pattern, cell)

    roads = []
    for rd, km in matches:
        rd_name = rd.strip()
        km_v = int(km) if km else None
        roads.append((rd_name, km_v))

    if not roads:
        return pd.Series([None, None, 0])

    roads_w_km = [r for r in roads if r[1] is not None]
    if roads_w_km:
        nrst = min(roads_w_km, key=lambda x: x[1])
        nrst_name = nrst[0]
        nrst_km = nrst[1]
    else:
        nrst_name = roads[0][0]
        nrst_km = None

    return pd.Series([nrst_name, nrst_km, len(roads)])

In [16]:
df[["nearest_road", "distance_from_mkad", "roads_nearby_count"]] = df["highway_info"].apply(parse_roads)

In [17]:
df["nearest_road"].unique()

array([nan, 'Калужское шоссе', 'Косинское шоссе', 'Варшавское шоссе',
       'Пятницкое шоссе', 'Дмитровское шоссе', 'Ленинградское шоссе',
       'Киевское шоссе', 'Новорижское шоссе', 'Сколковское шоссе',
       'Боровское шоссе', 'Носовихинское шоссе', 'Минское шоссе',
       'Куркинское шоссе', 'Щелковское шоссе', 'Алтуфьевское шоссе',
       'Волоколамское шоссе', 'Симферопольское шоссе',
       'Ярославское шоссе', 'Новорязанское шоссе', 'Рязанское шоссе',
       'Новоегорьевское шоссе', 'Каширское шоссе', 'Осташковское шоссе',
       'Каспий шоссе', 'Новосходненское шоссе', 'Можайское шоссе',
       'Рублевское шоссе', 'M-11 шоссе', 'Машкинское шоссе',
       'Рублево-Успенское шоссе'], dtype=object)

In [18]:
df[["distance_from_mkad", "nearest_road"]].dropna().head()

,distance_from_mkad,nearest_road
3,59.0,Калужское шоссе
4,5.0,Косинское шоссе
5,42.0,Калужское шоссе
7,27.0,Калужское шоссе
9,34.0,Варшавское шоссе


Приведем текстовые данные к стандартному формату.

In [19]:
def clean_text(cell):
    if not isinstance(cell, str):
        return cell
    cell = cell.strip()
    cell = cell.strip(" ")
    cell = cell.strip(",")
    cell = cell.lower()
    return cell

for col in df.select_dtypes(include = "object").columns:
    df[col] = df[col].apply(clean_text)

#Посчитаем описательные статистики

Так как задание не конкретизировано, выделим следующие описательные статистики - число и для числовых признаков:



1.   Доля_пропущенных

1.   Среднее

1.   Медиана

1.   Мода
2.   Отклонение (стандартное)


2.   Дисперсия


2.   Минимум


2.   Максимум


2.   Разброс минимума с максимумом (Разброс_мин_макс)


1.   Квантиль_25

1.   Квантиль_75

1.   Межквантильный_размах
2.   Коэффициент_вариации


2.   Ассиметрия


2.   Эксцесс

Для категориальных данных существует меньше статистик, выбраны следующие:


1.   Доля_пропущенных

1.   Доля_уникальных (из заполненных)
2.   Мода
3.   Доля_самого_частого_значения



In [26]:
def get_complete_stats(df):
    num_df = df.select_dtypes(include = [np.number])
    cat_df = df.select_dtypes(exclude = [np.number])
    total_rows = len(df)

    num_stats = pd.DataFrame()
    if len(num_df.columns) > 0:
        num_stats["Доля_пропущенных"] = num_df.isna().sum()/total_rows
        num_stats["Среднее"] = num_df.mean()
        num_stats["Медиана"] = num_df.median()
        try:
            num_stats["Мода"] = num_df.mode().iloc[0]
        except Exception:
            num_stats["Мода"] = np.nan
        num_stats["Отклонение"] = num_df.std()
        num_stats["Дисперсия"] = num_df.var()
        num_stats["Минимум"] = num_df.min()
        num_stats["Максимум"] = num_df.max()
        num_stats["Разброс_мин_макс"] = num_stats["Максимум"] - num_stats["Минимум"]
        num_stats["Квантиль_25"] = num_df.quantile(0.25)
        num_stats["Квантиль_75"] = num_df.quantile(0.75)
        num_stats["Межквартильный размах"] = num_stats["Квантиль_75"] - num_stats["Квантиль_25"]
        num_stats["Коэффициент_вариации"] = num_stats["Отклонение"]/num_stats["Среднее"]
        num_stats["Ассиметрия"] = num_df.skew()
        num_stats["Эксцесс"] = num_df.kurtosis()

    cat_stats = pd.DataFrame()
    if len(cat_df.columns) > 0:
        nan_cnt = cat_df.isna().sum()
        filled_cnt = total_rows - nan_cnt
        cat_stats["Доля_пропущенных"] = nan_cnt/total_rows
        cat_stats["Доля_уникальных"] = cat_df.nunique()/filled_cnt
        try:
            cat_stats["Мода"] = cat_df.mode().iloc[0]
        except Exception:
            cat_stats["Мода"] = np.nan
        top_part = []
        for col in cat_df.columns:
            s = cat_df[col].dropna()
            if len(s) == 0:
                top_part.append(0)
            else:
                top_part.append(s.value_counts(normalize = True).iloc[0])
        cat_stats["Доля_самого_частого_значения"] = top_part

    return num_stats, cat_stats

In [27]:
st=get_complete_stats(df)
pd.options.display.float_format = '{:.3f}'.format
st[0]

,Доля_пропущенных,Среднее,Медиана,Мода,Отклонение,Дисперсия,Минимум,Максимум,Разброс_мин_макс,Квантиль_25,Квантиль_75,Межквартильный размах,Коэффициент_вариации,Ассиметрия,Эксцесс
price,0.000,54197134.574,25000000.000,14500000.000,115573579.624,13357252307166628.000,850000.000,8174870000.000,8174020000.000,16378372.000,45204120.000,28825748.000,2.132,14.260,541.216
total_area,0.000,73.750,60.000,60.000,59.909,3589.054,6.000,4288.000,4282.000,41.800,83.500,41.700,0.812,10.055,435.540
living_area,0.243,39.180,31.400,20.000,34.868,1215.808,0.200,1150.000,1149.800,20.000,46.000,26.000,0.890,6.616,99.454
kitchen_area,0.184,14.350,12.000,10.000,9.577,91.727,0.100,240.000,239.900,8.000,18.400,10.400,0.667,2.766,19.824
ceiling_height,0.213,3.031,3.000,3.000,2.130,4.539,0.000,310.000,310.000,2.730,3.100,0.370,0.703,118.910,16512.116
floor,0.000,11.143,8.000,2.000,10.116,102.334,-2.000,84.000,86.000,4.000,15.000,11.000,0.908,1.939,5.114
total_floors,0.000,21.199,17.000,9.000,14.168,200.740,1.000,97.000,96.000,11.000,27.000,16.000,0.668,1.511,2.911
combined_bathroom,0.429,1.400,1.000,1.000,0.701,0.491,1.000,9.000,8.000,1.000,2.000,1.000,0.501,2.333,8.596
separated_bathroom,0.685,1.201,1.000,1.000,0.549,0.301,1.000,9.000,8.000,1.000,1.000,0.000,0.457,4.073,26.716
balcony,0.812,1.181,1.000,1.000,0.491,0.241,1.000,4.000,3.000,1.000,1.000,0.000,0.415,3.141,10.951


In [28]:
st[1]

,Доля_пропущенных,Доля_уникальных,Мода,Доля_самого_частого_значения
url,0.000,1.000,https://www.cian.ru/sale/flat/150959906/,0.000
address,0.000,0.261,"москва, зао, р-н можайский",0.005
auction,0.000,0.000,н,1.000
rooms_count,0.000,0.000,2,0.318
new_building,0.000,0.000,вторичка,0.592
apartment,0.884,0.000,апартаменты,0.957
view_from_the_windows,0.435,0.000,на улицу и двор,0.402
repair,0.443,0.000,дизайнерский,0.296
construction_series,0.627,0.004,индивидуальный проект,0.682
floor_type,0.635,0.000,железобетонные,0.964
